# 🚀 MultiModal RAG — Cloud Ingestion (Google Colab)

**Run this notebook on Colab to ingest PDFs into Qdrant Cloud.**
Your local machine then queries the same Qdrant Cloud collection for retrieval.

| Step | Where | Tool |
|------|-------|------|
| PDF Parsing | ☁️ Colab GPU | Ollama + glm-ocr |
| Image Captioning | ☁️ Colab | Groq (free) |
| Embeddings | ☁️ Colab GPU | sentence-transformers |
| Vector Storage | ☁️ Qdrant Cloud | cloud.qdrant.io (free) |
| Retrieval + Reranking | 🖥️ Local | BGE + FastAPI |

### Prerequisites
1. **Qdrant Cloud** free account → [cloud.qdrant.io](https://cloud.qdrant.io) (no credit card)
2. **Groq** free API key → [console.groq.com](https://console.groq.com)
3. Runtime → **Runtime > Change runtime type → T4 GPU**

> ⚠️ Fill in your keys in **Cell 1** before running anything else.

## 🔑 Cell 1 — Secrets & Config

Fill in your Qdrant Cloud URL, API key, and Groq API key below.

In [1]:
import os

# ── FILL THESE IN ────────────────────────────────────────────────────────────
QDRANT_API_KEY = 'your-qdrant-api-key'
QDRANT_URL = 'https://YOUR-CLUSTER-ID.cloud.qdrant.io'
QDRANT_COLLECTION = 'documents'

GROQ_API_KEY = "your_groq_api_key_here"
GROQ_VISION_MODEL = 'qwen/qwen3.6-27b'
GROQ_TEXT_MODEL   = 'llama-3.3-70b-versatile'

EMBEDDING_MODEL   = 'all-MiniLM-L6-v2'   # 384d, ~22 MB
EMBEDDING_DIMS    = 384
# ─────────────────────────────────────────────────────────────────────────────

# Validate — will error if you forgot to fill in the placeholders
assert 'YOUR-CLUSTER' not in QDRANT_URL, '❌ Set your Qdrant Cloud URL above'
assert 'your-qdrant' not in QDRANT_API_KEY, '❌ Set your Qdrant API key above'
assert 'your_groq' not in GROQ_API_KEY, '❌ Set your Groq API key above'

os.environ.update({
    'QDRANT_URL': QDRANT_URL,
    'QDRANT_API_KEY': QDRANT_API_KEY,
    'QDRANT_COLLECTION_NAME': QDRANT_COLLECTION,
    'GROQ_API_KEY': GROQ_API_KEY,
    'GROQ_VISION_MODEL': GROQ_VISION_MODEL,
    'GROQ_TEXT_MODEL': GROQ_TEXT_MODEL,
    'EMBEDDING_PROVIDER': 'local',
    'EMBEDDING_MODEL': EMBEDDING_MODEL,
    'EMBEDDING_DIMENSIONS': str(EMBEDDING_DIMS),
    'PARSER_BACKEND': 'ollama',
    'RERANKER_BACKEND': 'bge',
    'IMAGE_CAPTION_ENABLED': 'true',
    'LOG_LEVEL': 'INFO',
})

print('✅ Config set')
print(f'   Qdrant  : {QDRANT_URL}')
print(f'   Groq    : {GROQ_API_KEY[:8]}...')
print(f'   Embed   : {EMBEDDING_MODEL} ({EMBEDDING_DIMS}d)')

✅ Config set
   Qdrant  : https://YOUR-CLUSTER-ID.cloud.qdrant.io
   Groq    : your_gro...
   Embed   : all-MiniLM-L6-v2 (384d)


## 📦 Cell 2 — Install Dependencies

Clones the repo and installs all required packages. Takes ~2 minutes on first run.

In [2]:
import subprocess, sys

def run(cmd, **kw):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return r

# Clone the project
import os
# Define the consistent directory name (uppercase, aligning with os.chdir and sys.path)
repo_dir = 'MULTIMODAL_RAG'
if not os.path.exists(f'/content/{repo_dir}'):
    # Explicitly clone into the specified directory name to ensure consistency
    run(f'git clone https://github.com/SAIKIRANPATNANA/MULTIMODAL_RAG.git {repo_dir}')
    print('✅ Repo cloned')
else:
    print('✅ Repo already present')

os.chdir(f'/content/{repo_dir}')
sys.path.insert(0, f'/content/{repo_dir}/src')

# Install uv for fast package management
run('pip install uv -q')

# Install project with GPU extras
run('uv pip install -e ".[local-embed,bge,layout]" --system -q')

print('✅ All packages installed')

✅ Repo already present
$ pip install uv -q
$ uv pip install -e ".[local-embed,bge,layout]" --system -q
✅ All packages installed


In [3]:
run('git -C /content/MULTIMODAL_RAG pull origin main')
print('✅ Repo updated')

$ git -C /content/MULTIMODAL_RAG pull origin main
Already up to date.

✅ Repo updated


## 🦙 Cell 3 — Install Ollama & Pull glm-ocr

Installs Ollama on Colab, starts the server, and downloads the glm-ocr model (~2.2 GB).

In [4]:
import subprocess, time, urllib.request

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1000:] if r.stdout else '')
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
        raise RuntimeError(f'Command failed: {cmd}')
    return r

# Install zstd dependency for Ollama
print('Installing zstd dependency...')
run('sudo apt-get update -qq && sudo apt-get install -y zstd -qq')

# Install Ollama
print('Installing Ollama...')
run('curl -fsSL https://ollama.com/install.sh | sh')

# Start server in background
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Ollama server starting...')
time.sleep(3)

# Verify
try:
    urllib.request.urlopen('http://localhost:11434')
    print('✅ Ollama server running')
except Exception as e:
    print(f'⚠ Server not up yet, waiting... Error: {e}')
    time.sleep(5)
    # Try again after waiting
    try:
        urllib.request.urlopen('http://localhost:11434')
        print('✅ Ollama server running after retry')
    except Exception as e_retry:
        print(f'❌ Ollama server failed to start: {e_retry}')
        raise # Re-raise if still not running to stop execution

# Pull glm-ocr (2.2 GB — takes 2-5 min on Colab)
print('Pulling glm-ocr model (~2.2 GB)...')
run('ollama pull glm-ocr:latest')

# Confirm
r = run('ollama list')
print('✅ glm-ocr ready' if 'glm-ocr' in r.stdout else '✗ glm-ocr not found — check output above')

Installing zstd dependency...

Installing Ollama...

Ollama server starting...
✅ Ollama server running
Pulling glm-ocr model (~2.2 GB)...

NAME              ID              SIZE      MODIFIED               
glm-ocr:latest    6effedd0dc8a    2.2 GB    Less than a second ago    

✅ glm-ocr ready


## 📂 Cell 4 — Upload your PDF

Upload a PDF from your computer into Colab, or mount Google Drive to use existing files.

In [5]:
from google.colab import files as colab_files
from pathlib import Path
import os

# ── Option A: Upload from local computer ─────────────────────────────────────
# print('Select a PDF to upload (or skip and use Option B below):')
# uploaded = colab_files.upload()

# if uploaded:
#     PDF_PATH = Path(list(uploaded.keys())[0])
#     print(f'✅ Uploaded: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024:.1f} KB)')
# else:
#     # ── Option B: Mount Google Drive ─────────────────────────────────────────
#     # from google.colab import drive
#     # drive.mount('/content/drive')
#     # PDF_PATH = Path('/content/drive/MyDrive/your_document.pdf')
#     # print('No file uploaded. Set PDF_PATH manually below.')
PDF_PATH = Path('/content/attention_is_all_you_need.pdf')  # change this

if not PDF_PATH.exists():
    print(f'✗ File not found: {PDF_PATH}')
else:
    print(f'📄 Will parse: {PDF_PATH.name}')

📄 Will parse: attention_is_all_you_need.pdf


## 📄 Cell 5 — Parse PDF via Ollama + glm-ocr

First parse takes 30-60s (model warm-up), subsequent pages are faster.

In [8]:
import sys, os
import importlib
import subprocess, time, urllib.request # Added for Ollama checks

# Install glmocr package, a dependency for DocumentParser
print('Installing glmocr...')
!pip install glmocr -q
print('✅ glmocr installed.')

# --- Diagnostic: Verify glmocr is importable immediately after installation ---
try:
    import glmocr
    print('✅ glmocr is directly importable after installation.')
except ImportError as e:
    print(f'❌ glmocr is NOT directly importable after installation: {e}')
    # If this diagnostic fails, the problem is with the installation path or environment.
    # Raising an error to stop execution for further analysis.
    raise
# -----------------------------------------------------------------------------

# Ensure Ollama server is running before proceeding with DocumentParser
# This section is added to ensure the Ollama server is active.
print('Ensuring Ollama server is running...')
try:
    urllib.request.urlopen('http://localhost:11434')
    print('✅ Ollama server already running.')
except Exception:
    print('Ollama server not detected, restarting...')
    # Start server in background
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5) # Give it a bit more time to start
    try:
        urllib.request.urlopen('http://localhost:11434')
        print('✅ Ollama server restarted and running.')
    except Exception as e_retry:
        print(f'❌ Failed to restart Ollama server: {e_retry}')
        raise # Re-raise if still not running to stop execution


# NOW that glmocr is installed, make doc_parser discoverable and change directory
# This ensures doc_parser does not try to import glmocr before it's available.
sys.path.insert(0, '/content/MULTIMODAL_RAG/src')
os.chdir('/content/MULTIMODAL_RAG')

# Aggressive module cache clearing to force a fresh import of doc_parser and glmocr
# This is necessary if doc_parser modules were loaded before glmocr was installed,
# and their internal _GLMOCR_AVAILABLE flag was cached as False.
# This might be less critical with the new ordering, but kept for robustness.
print('Clearing doc_parser and glmocr modules from cache...')
for module_name in list(sys.modules.keys()):
    if module_name.startswith('doc_parser') or module_name == 'glmocr':
        del sys.modules[module_name]
print('✅ Modules cleared.')

# Import doc_parser modules *after* glmocr has been installed and cache cleared.
# This ensures that when doc_parser.pipeline is initialized, it correctly finds glmocr.
from doc_parser.config import get_settings, configure_logging
from doc_parser.pipeline import DocumentParser

configure_logging('INFO')
settings = get_settings()

print(f'Parsing {PDF_PATH.name} via Ollama glm-ocr...')
parser = DocumentParser()
result = parser.parse_file(PDF_PATH)

print(f'✅ Parsed: {len(result.pages)} pages, {result.total_elements} elements')
for p in result.pages:
    labels = set(e.label for e in p.elements)
    print(f'  Page {p.page_num}: {len(p.elements)} elements — {labels}')

Installing glmocr...


Starting Pipeline...


✅ glmocr installed.
✅ glmocr is directly importable after installation.
Ensuring Ollama server is running...
✅ Ollama server already running.
Clearing doc_parser and glmocr modules from cache...
✅ Modules cleared.
Parsing attention_is_all_you_need.pdf via Ollama glm-ocr...


Loading weights:   0%|          | 0/858 [00:00<?, ?it/s]

Pipeline started!
GLM-OCR initialized in self-hosted mode


✅ Parsed: 15 pages, 35 elements
  Page 1: 2 elements — {'paragraph_title', 'doc_title'}
  Page 2: 3 elements — {'paragraph_title'}
  Page 3: 3 elements — {'paragraph_title', 'image'}
  Page 4: 4 elements — {'paragraph_title', 'formula', 'image'}
  Page 5: 6 elements — {'paragraph_title', 'formula'}
  Page 6: 4 elements — {'paragraph_title', 'formula'}
  Page 7: 6 elements — {'paragraph_title', 'formula'}
  Page 8: 3 elements — {'paragraph_title'}
  Page 9: 1 elements — {'paragraph_title'}
  Page 10: 2 elements — {'paragraph_title'}
  Page 11: 0 elements — set()
  Page 12: 0 elements — set()
  Page 13: 0 elements — set()
  Page 14: 1 elements — {'image'}
  Page 15: 0 elements — set()


## 🧩 Cell 6 — Structure-Aware Chunking

In [9]:
from collections import Counter
from doc_parser.chunker import document_aware_chunking

pages = [(p.page_num, p.elements) for p in result.pages]
all_chunks = document_aware_chunking(pages, source_file=PDF_PATH.name)

counts = Counter(c.modality for c in all_chunks)
print(f'✅ {len(all_chunks)} chunks created: {dict(counts)}')
for c in all_chunks[:3]:
    print(f'  [{c.chunk_id}] {c.modality} | {c.text[:100]}...')

✅ 35 chunks created: {'text': 25, 'image': 3, 'formula': 7}
  [attention_is_all_you_need.pdf_1_0] text | #...
  [attention_is_all_you_need.pdf_1_1] text | ##...
  [attention_is_all_you_need.pdf_2_2] text | ##...


## 🖼️ Cell 7 — Caption Images/Tables via Groq

Vision model: `qwen/qwen3.6-27b` (images) | Text model: `meta/llama-3.3-70b-versatile` (tables/formulas)

In [10]:
# Force-reload image_captioner so git-pulled fixes take effect
import sys
for _m in list(sys.modules):
    if _m.startswith('doc_parser'):
        del sys.modules[_m]

from doc_parser.ingestion.image_captioner import enrich_chunks
from doc_parser.config import get_settings
settings = get_settings()

non_text = [c for c in all_chunks if c.modality != 'text']
print(f'Captioning {len(non_text)} non-text chunks (images, tables, formulas) via Groq...')

all_chunks = await enrich_chunks(
    all_chunks,
    pdf_path=PDF_PATH,
    settings=settings,
    max_concurrent=3,
)

captioned = [c for c in all_chunks if c.caption]
print(f'✅ {len(captioned)} chunks captioned')
for c in captioned[:3]:
    print(f'  [{c.modality}] {repr(c.caption[:200])}')

Stopping Pipeline...
Pipeline stopped!


Captioning 10 non-text chunks (images, tables, formulas) via Groq...
✅ 10 chunks captioned
  [image] "<think>\nThe user wants me to analyze a figure.\n\n**1. Classify the figure:**\nThe figure shows a block diagram with boxes, arrows, and text labels describing a neural network architecture. It's clearly "
  [image] '<think>\nThe user wants me to analyze a figure showing two diagrams related to attention mechanisms in neural networks.\n\n**1. Classify the figure:**\nThe figure contains block diagrams showing data flow'
  [formula] '$$\n\n$$'


## 🔢 Cell 8 — Embed (sentence-transformers, GPU-accelerated)

On a T4 GPU, embedding 100 chunks takes ~2 seconds vs ~20 seconds on CPU.

In [11]:
import torch
from doc_parser.ingestion.embedder import get_embedder, embed_chunks

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Force GPU for sentence-transformers if available
os.environ['SENTENCE_TRANSFORMERS_DEVICE'] = device

embedder = get_embedder(settings)
print(f'Embedding {len(all_chunks)} chunks with [{settings.embedding_model}] on {device}...')

dense, sparse = await embed_chunks(all_chunks, embedder, settings)

print(f'✅ Dense : {len(dense)} vectors × {len(dense[0])}d')
print(f'   Sparse: {len(sparse)} BM25 vectors')

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 35 chunks with [all-MiniLM-L6-v2] on cuda...
✅ Dense : 35 vectors × 384d
   Sparse: 35 BM25 vectors


## 🗄️ Cell 9 — Upsert to Qdrant Cloud

Writes all chunks to your Qdrant Cloud collection. Both Colab and your local machine can now query it!

In [12]:
from doc_parser.ingestion.vector_store import QdrantDocumentStore

print(f'Connecting to Qdrant Cloud: {settings.qdrant_url}')
store = QdrantDocumentStore(settings)

# overwrite=True means it skips creation if collection already exists
await store.create_collection(overwrite=True)

upserted = await store.upsert_chunks(all_chunks, dense, sparse)
print(f'✅ Upserted {upserted} points → collection "{settings.qdrant_collection_name}"')
print(f'   Dashboard: {settings.qdrant_url}/dashboard')
print()
print('📋 LOCAL MACHINE SETUP:')
print(f'   Set in your .env on local:')
print(f'   QDRANT_URL={QDRANT_URL}')
print(f'   QDRANT_API_KEY={QDRANT_API_KEY}')
print(f'   EMBEDDING_DIMENSIONS={EMBEDDING_DIMS}')
print(f'   Then run Cells 0-2 + Cell 10 in 01_quickstart.ipynb to search!')

Connecting to Qdrant Cloud: https://YOUR-CLUSTER-ID.cloud.qdrant.io
✅ Upserted 35 points → collection "documents"
   Dashboard: https://YOUR-CLUSTER-ID.cloud.qdrant.io/dashboard

📋 LOCAL MACHINE SETUP:
   Set in your .env on local:
   QDRANT_URL=https://YOUR-CLUSTER-ID.cloud.qdrant.io
   QDRANT_API_KEY=your-qdrant-api-key
   EMBEDDING_DIMENSIONS=384
   Then run Cells 0-2 + Cell 10 in 01_quickstart.ipynb to search!


## 🔍 Cell 10 — Quick Search Verification (Optional)

Run a quick search from Colab itself to verify everything was stored correctly.

In [15]:
QUERY = 'self attention'

embedder2 = get_embedder(settings)
candidates = await store.search(
    query_text=QUERY,
    embedder=embedder2,
    settings=settings,
    top_k=5,
)

print(f'Query: "{QUERY}"')
print(f'Retrieved {len(candidates)} results from Qdrant Cloud:\n')
for i, r in enumerate(candidates, 1):
    print(f'[{i}] modality={r.get("modality")}  page={r.get("page")}')
    print(f'    {r.get("text","")}')
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: "self attention"
Retrieved 5 results from Qdrant Cloud:

[1] modality=image  page=4
    <think>
The user wants me to analyze a figure showing two diagrams related to attention mechanisms in neural networks.

**1. Classify the figure:**
The figure contains block diagrams showing data flow and operations. It's clearly a DIAGRAM.

**2. Analyze the figure:**
- **Left Diagram:** Titled "Scaled Dot-Product Attention" (partially cut off but recognizable).
    - Inputs: Q, K, V at the bottom.
    - Operations (bottom to top):
        - MatMul (Q and K go in).
        - Scale.
        - Mask (opt.) - optional masking.
        - SoftMax.
        - MatMul (output of SoftMax and V go in).
    - Output: Arrow pointing up.
- **Right Diagram:** Titled "Multi-Head Attention".
    - Inputs: V, K, Q at the bottom.
    - Operations:
        - Three "Linear" layers at the bottom, taking V, K, Q respectively.
        - These feed into a block labeled "Scaled Dot-Product Attention". This block is sho